In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras import layers
from tokenizers import ByteLevelBPETokenizer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import kagglehub

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
adityajn105_flickr8k_path = kagglehub.dataset_download('adityajn105/flickr8k')

CONFIG = {
    "image_size"  : 224,
    "patch_size"  : 16,
    "num_patches" : 196,
    "D"           : 768,
    "num_heads"   : 12,
    "num_layers"  : 6,
    "mlp_dim"     : 3072,
    "dropout"     : 0.1,
    "vocab_size"  : 10000,
    "max_len"     : 40,
    "batch_size"  : 32,
    "epochs"      : 20,
    "save_path"   : "captioning_model.keras"
}

captions_path = os.path.join(adityajn105_flickr8k_path, 'captions.txt')
images_path   = os.path.join(adityajn105_flickr8k_path, 'Images')
print("Captions path:", captions_path)
print("Images path  :", images_path)

In [ ]:
def image_preprocessing(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (CONFIG["image_size"], CONFIG["image_size"]))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype(np.float32) / 255.0
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    image = (image - mean) / std
    return image

def create_patches(image, patch_size=16):
    H, W, C = image.shape
    patches = []
    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            patch = image[i:i+patch_size, j:j+patch_size, :]
            patch = patch.reshape(-1)
            patches.append(patch)
    return np.array(patches)

def load_captions(captions_path):
    df = pd.read_csv(captions_path)
    df.columns = df.columns.str.strip().str.lower()
    df["image"]   = df["image"].apply(lambda x: os.path.basename(x).split("#")[0].strip())
    df["caption"] = df["caption"].apply(lambda x: "<start> " + str(x).strip().lower() + " <end>")
    captions_dict = {}
    for _, row in df.iterrows():
        img_name = row["image"]
        caption  = row["caption"]
        if img_name not in captions_dict:
            captions_dict[img_name] = []
        captions_dict[img_name].append(caption)
    return captions_dict

def get_all_pairs(captions_dict):
    image_paths = []
    captions    = []
    for img_name, caps in captions_dict.items():
        full_path = os.path.join(images_path, img_name)
        if os.path.exists(full_path):
            for cap in caps:
                image_paths.append(full_path)
                captions.append(cap)
    return image_paths, captions

captions_dict             = load_captions(captions_path)
all_img_paths, all_caps_list = get_all_pairs(captions_dict)
print("Total pairs:", len(all_img_paths))
print("Sample caption:", all_caps_list[0])

In [ ]:
with open("captions_corpus.txt", "w", encoding="utf-8") as f:
    for caption in all_caps_list:
        f.write(caption + "\n")

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=["captions_corpus.txt"],
    vocab_size=CONFIG["vocab_size"],
    min_frequency=2,
    special_tokens=["<pad>", "<start>", "<end>", "<unk>"]
)
tokenizer.save_model(".", "caption_tokenizer")

tokenizer = ByteLevelBPETokenizer(
    "caption_tokenizer-vocab.json",
    "caption_tokenizer-merges.txt"
)
tokenizer.add_special_tokens(["<pad>", "<start>", "<end>", "<unk>"])

CONFIG["vocab_size"] = tokenizer.get_vocab_size()
pad_id               = tokenizer.token_to_id("<pad>")
start_id             = tokenizer.token_to_id("<start>")
end_id               = tokenizer.token_to_id("<end>")

print("Vocab size:", CONFIG["vocab_size"])
print("pad_id    :", pad_id)
print("start_id  :", start_id)
print("end_id    :", end_id)

In [ ]:
def tokenize_caption(caption):
    ids    = tokenizer.encode(caption).ids[:CONFIG["max_len"]]
    padded = ids + [0] * (CONFIG["max_len"] - len(ids))
    return padded

all_caps_tokenized = np.array(
    [tokenize_caption(cap) for cap in all_caps_list], dtype=np.int32
)

def load_data_fn(img_path, caption_ids):
    img     = tf.io.read_file(img_path)
    img     = tf.image.decode_jpeg(img, channels=3)
    img     = tf.image.resize(img, (224, 224))
    img     = (tf.cast(img, tf.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]

    patches = tf.image.extract_patches(
        images=tf.expand_dims(img, 0),
        sizes=[1, CONFIG['patch_size'], CONFIG['patch_size'], 1],
        strides=[1, CONFIG['patch_size'], CONFIG['patch_size'], 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    patches = tf.reshape(patches, (CONFIG['num_patches'], -1))

    input_cap  = caption_ids[:-1]
    target_cap = caption_ids[1:]

    return (patches, input_cap), target_cap

dataset = tf.data.Dataset.from_tensor_slices((all_img_paths, all_caps_tokenized))
dataset = dataset.shuffle(1000).map(load_data_fn, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(CONFIG['batch_size']).prefetch(tf.data.AUTOTUNE)

total      = len(all_img_paths)
train_size = int(total * 0.8)
val_size   = int(total * 0.1)

train_dataset = dataset.take(train_size // CONFIG['batch_size'])
val_dataset   = dataset.skip(train_size // CONFIG['batch_size']).take(val_size // CONFIG['batch_size'])
test_dataset  = dataset.skip((train_size + val_size) // CONFIG['batch_size'])

CONFIG['max_len_input'] = CONFIG['max_len'] - 1

In [ ]:
class ViT(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ViT, self).__init__(**kwargs)
        self.patch_embed    = layers.Dense(CONFIG['D'])
        self.cls_token      = layers.Embedding(1, CONFIG['D'])
        self.pos_embedding  = layers.Embedding(CONFIG['num_patches'] + 1, CONFIG['D'])
        self.attention_layers = [
            layers.MultiHeadAttention(
                num_heads=CONFIG['num_heads'],
                key_dim=CONFIG['D'] // CONFIG['num_heads'],
                dropout=CONFIG['dropout']
            ) for _ in range(CONFIG['num_layers'])
        ]
        self.mlp_layers = [
            tf.keras.Sequential([
                layers.Dense(CONFIG['mlp_dim'], activation='gelu'),
                layers.Dropout(CONFIG['dropout']),
                layers.Dense(CONFIG['D']),
                layers.Dropout(CONFIG['dropout'])
            ]) for _ in range(CONFIG['num_layers'])
        ]
        self.norm1      = [layers.LayerNormalization(epsilon=1e-6) for _ in range(CONFIG['num_layers'])]
        self.norm2      = [layers.LayerNormalization(epsilon=1e-6) for _ in range(CONFIG['num_layers'])]
        self.norm_final = layers.LayerNormalization(epsilon=1e-6)

    def call(self, patches, training=False):
        patches = tf.ensure_shape(patches, [None, CONFIG['num_patches'], CONFIG['D']])
        B = tf.shape(patches)[0]

        x = self.patch_embed(patches)

        cls_tokens = self.cls_token(tf.zeros((B, 1), dtype=tf.int32))
        x = tf.concat([cls_tokens, x], axis=1)

        pos_indices = tf.range(CONFIG['num_patches'] + 1)
        x = x + self.pos_embedding(pos_indices)

        for i in range(CONFIG['num_layers']):
            attn_out = self.attention_layers[i](x, x)
            x = self.norm1[i](x + attn_out, training=training)
            mlp_out = self.mlp_layers[i](x)
            x = self.norm2[i](x + mlp_out, training=training)

        return self.norm_final(x)

    def get_config(self):
        config = super().get_config()
        config.update(CONFIG)
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
encoder = ViT()
for (patches, input_caps), target_caps in train_dataset.take(1):
    output = encoder(patches)
    print("Encoder input patches shape :", patches.shape)
    print("Encoder output shape        :", output.shape)

In [ ]:
class GPT(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(GPT, self).__init__(**kwargs)

        self.token_embedding = layers.Embedding(CONFIG['vocab_size'], CONFIG['D'])
        self.pos_embedding   = layers.Embedding(CONFIG['max_len'], CONFIG['D'])

        self.masked_attention = [
            layers.MultiHeadAttention(
                num_heads=CONFIG['num_heads'],
                key_dim=CONFIG['D'] // CONFIG['num_heads'],
                dropout=CONFIG['dropout']
            )
            for _ in range(CONFIG['num_layers'])
        ]
        self.cross_attention = [
            layers.MultiHeadAttention(
                num_heads=CONFIG['num_heads'],
                key_dim=CONFIG['D'] // CONFIG['num_heads'],
                dropout=CONFIG['dropout']
            )
            for _ in range(CONFIG['num_layers'])
        ]
        self.ffn = [
            tf.keras.Sequential([
                layers.Dense(CONFIG['mlp_dim'], activation='gelu'),
                layers.Dropout(CONFIG['dropout']),
                layers.Dense(CONFIG['D']),
                layers.Dropout(CONFIG['dropout'])
            ])
            for _ in range(CONFIG['num_layers'])
        ]
        self.norm1      = [layers.LayerNormalization(epsilon=1e-6) for _ in range(CONFIG['num_layers'])]
        self.norm2      = [layers.LayerNormalization(epsilon=1e-6) for _ in range(CONFIG['num_layers'])]
        self.norm3      = [layers.LayerNormalization(epsilon=1e-6) for _ in range(CONFIG['num_layers'])]
        self.norm_final = layers.LayerNormalization(epsilon=1e-6)
        self.output_layer = layers.Dense(CONFIG['vocab_size'])

    def call(self, caption_ids, encoder_output, training=False):
        seq_len = tf.shape(caption_ids)[1]
        x       = self.token_embedding(caption_ids)
        x       = x + self.pos_embedding(tf.range(seq_len))

        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        mask = (1 - mask) * -1e9

        for i in range(CONFIG['num_layers']):
            x_norm = self.norm1[i](x, training=training)
            attn   = self.masked_attention[i](x_norm, x_norm, attention_mask=mask, training=training)
            x      = x + attn

            x_norm = self.norm2[i](x, training=training)
            cross  = self.cross_attention[i](x_norm, encoder_output, training=training)
            x      = x + cross

            x_norm = self.norm3[i](x, training=training)
            ffn    = self.ffn[i](x_norm, training=training)
            x      = x + ffn

        x = self.norm_final(x)
        return self.output_layer(x)

    def get_config(self):
        config = super().get_config()
        config.update(CONFIG)
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
decoder = GPT()
for (patches, input_caps), target_caps in train_dataset.take(1):
    encoder_output = encoder(patches)
    logits         = decoder(input_caps, encoder_output)
    print("Patches shape  :", patches.shape)
    print("Decoder input  :", input_caps.shape)
    print("Logits shape   :", logits.shape)
    print("Targets shape  :", target_caps.shape)

In [ ]:
class ImageCaptioningModel(tf.keras.Model):
    def __init__(self, **kwargs):
        super(ImageCaptioningModel, self).__init__(**kwargs)
        self.encoder = ViT()
        self.decoder = GPT()

    def call(self, inputs, training=False):
        patches     = inputs[0]
        caption_ids = inputs[1]

        encoder_output = self.encoder(patches, training=training)
        logits         = self.decoder(caption_ids, encoder_output, training=training)
        return logits

    def get_config(self):
        config = super().get_config()
        config.update(CONFIG)
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
model = ImageCaptioningModel()

for (patches, input_caps), target_caps in train_dataset.take(1):
    logits = model((patches, input_caps))
    print("Patches shape   :", patches.shape)
    print("Input caps shape:", input_caps.shape)
    print("Logits shape    :", logits.shape)
    print("Targets shape   :", target_caps.shape)

In [ ]:
CONFIG["learning_rate"] = 1e-4
PAD_ID = pad_id

class MaskedSparseCategoricalCrossentropy(tf.keras.losses.Loss):
    def __init__(self, pad_id, name="masked_sparse_categorical_crossentropy"):
        super().__init__(name=name)
        self.pad_id  = pad_id
        self.loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction=None
        )

    def call(self, y_true, y_pred):
        mask = tf.math.logical_not(tf.math.equal(y_true, self.pad_id))
        loss = self.loss_fn(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

os.makedirs("checkpoints", exist_ok=True)

model = ImageCaptioningModel(name="image_captioning")

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=CONFIG["learning_rate"],
        clipnorm=1.0,
    ),
    loss=MaskedSparseCategoricalCrossentropy(pad_id=PAD_ID),
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath="checkpoints/epoch_{epoch:02d}_valloss_{val_loss:.4f}.keras",
        monitor="val_loss",
        save_best_only=True,
        save_freq="epoch",
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3,
    callbacks=callbacks,
)

In [ ]:
model.predict(test_dataset.take(1))

In [ ]:
# ✅ FIXED: generate_caption — no padding during inference
def generate_caption(image_path, temperature=1.0, use_sampling=False):
    """
    Generate a caption for the given image.

    Fix applied:
    - Original code padded the sequence to max_len_input before passing to the model.
      This caused the causal mask to attend to padding positions, corrupting the logits
      and producing empty/garbage output.
    - Fixed by passing ONLY the tokens generated so far (no padding) on each step.
      The GPT decoder already uses dynamic seq_len = tf.shape(caption_ids)[1],
      so variable-length inputs are fully supported.

    Args:
        image_path   : path to the image file
        temperature  : sampling temperature (only used when use_sampling=True)
        use_sampling : if True, sample from distribution; if False, use greedy argmax
    """
    # 1. Preprocess image
    img        = tf.io.read_file(image_path)
    img        = tf.image.decode_jpeg(img, channels=3)
    img_display = tf.image.resize(img, (224, 224)) / 255.0
    img_input  = (tf.cast(img_display, tf.float32) - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]

    patches = tf.image.extract_patches(
        images=tf.expand_dims(img_input, 0),
        sizes=[1, CONFIG['patch_size'], CONFIG['patch_size'], 1],
        strides=[1, CONFIG['patch_size'], CONFIG['patch_size'], 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    patches = tf.reshape(patches, (1, CONFIG['num_patches'], -1))

    # 2. Auto-regressive decoding — NO padding, grow sequence one token at a time
    decoded_caption = [start_id]

    for _ in range(CONFIG['max_len'] - 1):
        current_len = len(decoded_caption)

        # ✅ KEY FIX: pass only real tokens, no padding
        caption_tensor = tf.expand_dims(
            tf.convert_to_tensor(decoded_caption, dtype=tf.int32), 0
        )  # shape: (1, current_len)

        logits = model((patches, caption_tensor), training=False)

        # Take logits at the last generated position
        next_token_logits = logits[0, current_len - 1, :].numpy()

        # Suppress special tokens
        next_token_logits[pad_id]   = -1e9
        next_token_logits[start_id] = -1e9
        if current_len < 3:
            # Avoid ending too early
            next_token_logits[end_id] = -1e9

        # Greedy or temperature sampling
        if use_sampling:
            probs      = tf.nn.softmax(next_token_logits / temperature).numpy()
            prediction = int(np.random.choice(len(probs), p=probs))
        else:
            prediction = int(np.argmax(next_token_logits))

        if prediction == end_id:
            break

        decoded_caption.append(prediction)

    # 3. Decode tokens → string (skip <start>)
    result = tokenizer.decode(decoded_caption[1:])

    # 4. Display
    plt.figure(figsize=(6, 6))
    plt.imshow(img_display)
    plt.title(f"Generated: {result}", fontsize=12, wrap=True)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return result


# Test on a random image from the test split
test_img_path = np.random.choice(all_img_paths[train_size + val_size:])
caption = generate_caption(test_img_path)
print("Caption:", caption)

In [ ]:
# Optional: run on multiple test images to inspect quality
sample_paths = np.random.choice(all_img_paths[train_size + val_size:], size=5, replace=False)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, img_path in zip(axes, sample_paths):
    img        = tf.io.read_file(img_path)
    img        = tf.image.decode_jpeg(img, channels=3)
    img_display = tf.image.resize(img, (224, 224)) / 255.0
    caption    = generate_caption(img_path)
    ax.imshow(img_display)
    ax.set_title(caption, fontsize=8, wrap=True)
    ax.axis('off')

plt.tight_layout()
plt.show()